# Project Contributors

Inês Honrado, Jude Gbenimako, Ruben Marques, Susana Reis, and Tomás Carvalho.

*NOVA IMS — Machine Learning Group Project (2024/2025).*

# Import the Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.stats import skew
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import LabelEncoder
import category_encoders as ce
from sklearn.preprocessing import MinMaxScaler

from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LassoCV

import warnings
warnings.filterwarnings('ignore')

import joblib

# Context

Workers' compensation for workplace injuries in New York are administrated through the Workers' Compensation Board. Due to the fact that over 5 million claims have been processed since the year 2000, assessing claims manually is very time-consuming. The purpose of this project is to apply machine learning models, taking claims from 2020 to 2022, to conduct automated claim evaluations in order to seep up decision-making on new claims and increase WCB efficiency in handling compensation processes.

# Metadata

- **Accident Date**: Injury date of the claim.
- **Age at Injury**: Age of injured worker when the injury occurred.
- **Alternative Dispute Resolution**: Adjudication processes external to the Board.
- **Assembly Date**: The date the claim was first assembled.
- **Attorney/Representative**: Whether the claim is being represented by an attorney.
- **Average Weekly Wage**: The wage used to calculate workers’ compensation, disability, or paid leave wage replacement benefits.
- **Birth Year**: The reported year of birth of the injured worker.
- **C-2 Date**: Date of receipt of the Employer's Report of Work-Related Injury/Illness or equivalent (formerly Form C-2).
- **C-3 Date**: Date Form C-3 (Employee Claim Form) was received.
- **Carrier Name**: Name of the primary insurance provider responsible for providing workers’ compensation coverage to the injured worker’s employer.
- **Carrier Type**: Type of primary insurance provider responsible for providing workers’ compensation coverage.
- **Claim Identifier**: Unique identifier for each claim, assigned by WCB.
- **County of Injury**: Name of the New York County where the injury occurred.
- **COVID-19 Indicator**: Indicates if the claim may be associated with COVID-19.
- **District Name**: Name of the WCB district office that oversees claims for that region or area of the state.
- **First Hearing Date**: Date the first hearing was held on a claim at a WCB hearing location. A blank date means the claim has not yet had a hearing.
- **Gender**: The reported gender of the injured worker.
- **IME-4 Count**: Number of IME-4 forms received per claim. The IME-4 form is the "Independent Examiner's Report of Independent Medical Examination."
- **Industry Code**: NAICS code and descriptions are available at [NAICS Search](https://www.naics.com/search-naics-codes-by-industry/).
- **Industry Code Description**: 2-digit NAICS industry code description used to classify businesses according to their economic activity.
- **Medical Fee Region**: Approximate region where the injured worker would receive medical services.
- **OIICS Nature of Injury Description**: The OIICS nature of injury codes & descriptions are available at [BLS OIICS Manual](https://www.bls.gov/iif/oiics_manual_2007.pdf).
- **WCIO Cause of Injury Code**: The WCIO cause of injury codes & descriptions are available at [WCIO Cause Table](https://www.wcio.org/Active%20PNC/WCIO_Cause_Table.pdf).
- **WCIO Cause of Injury Description**: See description of field above.
- **WCIO Nature of Injury Code**: The WCIO nature of injury codes & descriptions are available at [WCIO Nature Table](https://www.wcio.org/Active%20PNC/WCIO_Nature_Table.pdf).
- **WCIO Nature of Injury Description**: See description of field above.
- **WCIO Part of Body Code**: The WCIO part of body codes & descriptions are available at [WCIO Part Table](https://www.wcio.org/Active%20PNC/WCIO_Part_Table.pdf).
- **WCIO Part of Body Description**: See description of field above.
- **Zip Code**: The reported ZIP code of the injured worker’s home address.
- **Agreement Reached**: Binary variable: Yes if there is an agreement without the involvement of the WCB (unknown at the start of a claim).
- **WCB Decision**: Multiclass variable: Decision of the WCB relative to the claim: "Accident" means that the claim refers to a workplace accident; "Occupational Disease" means illness from the workplace; requires WCB deliberation, so it is unknown at the start of a claim.
- **Claim Injury Type**: Main target variable: Deliberation of the WCB relative to benefits awarded to the claim. Numbering indicates severity.

# Import the data

In [ ]:
df_train = pd.read_csv('train_data.csv')
df_test = pd.read_csv('test_data.csv')

# Initial Analysis

In [ ]:
df_train.shape

We have 33 variables and 593471 observations.

In [ ]:
df_train.info()

The variable 'Claim Identifier' works has an ID, and is the perfect one to set as index.

In [ ]:
keep_identifier = df_test['Claim Identifier'].copy()

In [ ]:
df_train.set_index('Claim Identifier', inplace=True)
df_test.set_index('Claim Identifier', inplace=True)

**Checking duplicates:**

In [ ]:
duplicates = df_train.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

In [ ]:
df_train = df_train.drop_duplicates()

In [ ]:
duplicates = df_train.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

**Checking statistics:**

In [ ]:
df_train.describe()

**What we can observe:**
  - Values like 0 or 117 in 'Age at Injury', which indicate data errors.
  - There are a lot of 0 values in Average Weekly Wage.
  - The mean birth year is approximately 1886, which might be skewed by invalid entries such as 0.
  - The column 'OIICS Nature of Injury Description' is entirely missing, which might justify excluding it.
  - Given that most values in 'Agreement Reached' are zero, a large portion of claims may not have reached an agreement.

In [ ]:
df_train.describe(include='object')

**Correlations:**

In [ ]:
numerical_col = df_train.select_dtypes(include=['float64', 'int64']).columns

figure = plt.figure(figsize=(12, 6))
sns.heatmap(df_train[numerical_col].drop(columns = 'OIICS Nature of Injury Description').corr(method='spearman'), annot=True)

plt.xlabel('Corr')
plt.show()

There is a strong negative correlation between Age at Injury and Birth Year in the correlation heat map. This was expected since Birth Year and Age at Injury are intrinsically connected with one another -for each claim, Birth Year will be related to, and therefore determine, the Age at Injury.

In feature selection, based on this dependency, we would be better off removing one of these variables to avoid duplication and, consequently, any multicollinearity issues that might arise. 

# Clean and Pre-Process the Data

## Transform dates into datetime

In [ ]:
date_columns = ['Accident Date', 'Assembly Date', 'C-2 Date', 'C-3 Date', 'First Hearing Date']

for col in date_columns:
        df_train[col] = pd.to_datetime(df_train[col], errors='coerce')

for col in date_columns:
        df_test[col] = pd.to_datetime(df_test[col], errors='coerce')

## Drop null targets

In [ ]:
df_train['Claim Injury Type'].isnull().sum()

In [ ]:
df_train.dropna(subset=['Claim Injury Type'], inplace=True)

## Split df_train

We split the dataset into training and validation sets using the **holdout method**. This will allow us to train the model on one portion of the data (the training set) and evaluate its performance on another (validation set).

- **test_size=0.2**: 20% of the data is used for validation, while 80% is used for training.
- **random_state=42**: This ensures reproducibility of the split across different runs.
- **stratify=y**: The split is done while preserving the distribution of the target variable (`y`) in both the training and validation sets.
- **shuffle=True**: The data is shuffled before splitting, preventing any potential bias from the ordering of the data.

In [ ]:
target = 'Claim Injury Type'
X = df_train.drop(columns=[target])
y = df_train[target]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y, 
    shuffle=True)

In [ ]:
y_train.value_counts()

We have an imbalaced train data.

## Outliers

### Age at Injury

In [ ]:
plt.boxplot(X_train['Age at Injury'])
plt.title('Box Plot of Age at Injury')
plt.ylabel('Ages')
plt.show()

In [ ]:
print(X_train['Age at Injury'].describe())

Q1 = X_train['Age at Injury'].quantile(0.25)
Q3 = X_train['Age at Injury'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['Age at Injury'] < lower_bound) | (X_train['Age at Injury'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In the context of workplace, we should treat values below 16 and above 85 as outliers or invalid.

In [ ]:
def clean_age_column(df, age_col='Age at Injury', min_age=16, max_age=85):

    df[age_col] = df[age_col].apply(lambda x: x if min_age <= x <= max_age else np.nan)
    df[age_col] = df[age_col].astype('Int64')
    
    return df

In [ ]:
X_train = clean_age_column(X_train, age_col='Age at Injury', min_age=16, max_age=85)
X_val = clean_age_column(X_val, age_col='Age at Injury', min_age=16, max_age=85)

df_test = clean_age_column(df_test, age_col='Age at Injury', min_age=16, max_age=85)

In [ ]:
age_cleaned = X_train['Age at Injury'].dropna()
plt.boxplot(age_cleaned)
plt.title('Box Plot of Age at Injury (Cleaned)')
plt.ylabel('Ages')
plt.show()

### Average Weekly Wage

In [ ]:
aww_cleaned = X_train['Average Weekly Wage'].dropna()
plt.boxplot(aww_cleaned)
plt.title('Box Plot of Average Weekly Wage')
plt.ylabel('Wages')
plt.show()

In [ ]:
print(X_train['Average Weekly Wage'].describe())

Q1 = X_train['Average Weekly Wage'].quantile(0.25)
Q3 = X_train['Average Weekly Wage'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['Average Weekly Wage'] < lower_bound) | (X_train['Average Weekly Wage'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['Average Weekly Wage'].describe())

lower_percentile = X_train['Average Weekly Wage'].quantile(0.03)
upper_percentile = X_train['Average Weekly Wage'].quantile(0.97)

print(f'Lower Bound (2nd percentile): {lower_percentile}, Upper Bound (98th percentile): {upper_percentile}')

outliers = X_train[(X_train['Average Weekly Wage'] < lower_percentile) | (X_train['Average Weekly Wage'] > upper_percentile)]
print(f'Number of Outliers: {outliers.shape[0]}')

The 3rd and 97th percentile approach seems more resonable, and the bounds it sets make much more sense.

In [ ]:
def clean_aww_column(df, aww_col='Average Weekly Wage', min_aww=0, max_aww=2288): 
    
    df[aww_col] = df[aww_col].apply(lambda x: x if min_aww < x <= max_aww else np.nan)
    
    return df

In [ ]:
X_train = clean_aww_column(X_train, aww_col='Average Weekly Wage', min_aww=0, max_aww=2288)
X_val = clean_aww_column(X_val, aww_col='Average Weekly Wage', min_aww=0, max_aww=2288)
df_test = clean_aww_column(df_test, aww_col='Average Weekly Wage', min_aww=0, max_aww=2288)

In [ ]:
print(X_train['Average Weekly Wage'].describe())

In [ ]:
aww_cleaned = X_train['Average Weekly Wage'].dropna()
plt.boxplot(aww_cleaned)
plt.title('Box Plot of Average Weekly Wage')
plt.ylabel('Wages')
plt.show()

### Birth Year

In [ ]:
birth_year_cleaned = X_train['Birth Year'].dropna()

plt.boxplot(birth_year_cleaned)
plt.title('Box Plot of Birth Year')
plt.ylabel('Birth Year')
plt.show()

In [ ]:
print(X_train['Birth Year'].describe())

Q1 = X_train['Birth Year'].quantile(0.25)
Q3 = X_train['Birth Year'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['Birth Year'] < lower_bound) | (X_train['Birth Year'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In the context of workplace, we should treat ages below 16 and above 85 as outliers or invalid. So, birth years below 1939 (2024-85) or above 2008 (2024-16) should be considered outliers/invalid. 

In [ ]:
def clean_birth_year_column(df, birth_year_col='Birth Year', min_year=1939, max_year=2008):

    df[birth_year_col] = df[birth_year_col].apply(lambda x: x if (pd.notna(x) and min_year <= x <= max_year) else np.nan)
    
    return df 

In [ ]:
X_train = clean_birth_year_column(X_train, birth_year_col='Birth Year', min_year=1939, max_year=2008)
X_val = clean_birth_year_column(X_val, birth_year_col='Birth Year', min_year=1939, max_year=2008)
df_test = clean_birth_year_column(df_test, birth_year_col='Birth Year', min_year=1939, max_year=2008)

In [ ]:
birth_year_cleaned = X_train['Birth Year'].dropna()

plt.boxplot(birth_year_cleaned)
plt.title('Box Plot of Birth Year')
plt.ylabel('Birth Year')
plt.show()

### IME-4 Count

In [ ]:
ime4_count_cleaned = X_train['IME-4 Count'].dropna()

plt.boxplot(ime4_count_cleaned)
plt.title('Box Plot of IME-4 Count')
plt.ylabel('IME-4 Count')
plt.show()

In [ ]:
print(X_train['IME-4 Count'].describe())

Q1 = X_train['IME-4 Count'].quantile(0.25)
Q3 = X_train['IME-4 Count'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['IME-4 Count'] < lower_bound) | (X_train['IME-4 Count'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['IME-4 Count'].describe())

lower_percentile = X_train['IME-4 Count'].quantile(0.03)
upper_percentile = X_train['IME-4 Count'].quantile(0.97)

print(f'Lower Bound (3rd percentile): {lower_percentile}, Upper Bound (97th percentile): {upper_percentile}')

outliers = X_train[(X_train['IME-4 Count'] < lower_percentile) | (X_train['IME-4 Count'] > upper_percentile)]

print(f'Number of Outliers: {outliers.shape[0]}')

Again, the 3rd and 97th percentile approach seems more resonable, and the bounds it sets make much more sense.

In [ ]:
def clean_ime4_count_column(df, ime4_count_col='IME-4 Count', min_value=1, max_value=10):

    df[ime4_count_col] = df[ime4_count_col].apply(lambda x: x if (pd.notna(x) and min_value <= x <= max_value) else np.nan)
    
    return df

In [ ]:
X_train = clean_ime4_count_column(X_train, ime4_count_col='IME-4 Count')
X_val = clean_ime4_count_column(X_val, ime4_count_col='IME-4 Count')
df_test = clean_ime4_count_column(df_test, ime4_count_col='IME-4 Count')

In [ ]:
ime4_count_cleaned = X_train['IME-4 Count'].dropna()

plt.boxplot(ime4_count_cleaned)
plt.title('Box Plot of IME-4 Count')
plt.ylabel('IME-4 Count')
plt.show()

### Number of Dependents

In [ ]:
dependents_cleaned = X_train['Number of Dependents'].dropna()

plt.boxplot(dependents_cleaned)
plt.title('Box Plot of Number of Dependents')
plt.ylabel('Number of Dependents')
plt.show()

In [ ]:
print(X_train['Number of Dependents'].describe())

Q1 = X_train['Number of Dependents'].quantile(0.25)
Q3 = X_train['Number of Dependents'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['Number of Dependents'] < lower_bound) | (X_train['Number of Dependents'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['Number of Dependents'].describe())

lower_percentile = X_train['Number of Dependents'].quantile(0.03)
upper_percentile = X_train['Number of Dependents'].quantile(0.97)

print(f'Lower Bound (3rd percentile): {lower_percentile}, Upper Bound (97th percentile): {upper_percentile}')

outliers = X_train[(X_train['Number of Dependents'] < lower_percentile) | (X_train['Number of Dependents'] > upper_percentile)]

print(f'Number of Outliers: {outliers.shape[0]}')

There are no outliers.

### Accident Date

In [ ]:
plt.figure(figsize=(14, 6))
plt.hist(X_train['Accident Date'], bins=100, color='skyblue', edgecolor='black')
plt.title('Histogram of Accident Dates', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45) 
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(X_train['Accident Date'].describe())

Q1 = X_train['Accident Date'].quantile(0.25)
Q3 = X_train['Accident Date'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['Accident Date'] < lower_bound) | (X_train['Accident Date'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['Accident Date'].describe())

lower_percentile = X_train['Accident Date'].quantile(0.01)
upper_percentile = X_train['Accident Date'].quantile(0.99)

print(f'Lower Bound (1st percentile): {lower_percentile}, Upper Bound (99th percentile): {upper_percentile}')

outliers = X_train[(X_train['Accident Date'] < lower_percentile) | (X_train['Accident Date'] > upper_percentile)]

print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
def clean_accident_date_column(df, lower_percentile, upper_percentile, accident_date_col='Accident Date'):

    df[accident_date_col] = df[accident_date_col].apply(lambda x: x if (pd.notna(x) and lower_percentile <= x <= upper_percentile) else np.nan)
    
    return df

In [ ]:
X_train = clean_accident_date_column(X_train, lower_percentile, upper_percentile, accident_date_col='Accident Date')
X_val = clean_accident_date_column(X_val, lower_percentile, upper_percentile, accident_date_col='Accident Date')

In [ ]:
X_train['Accident Date'].describe()

### Assembly Date

In [ ]:
plt.figure(figsize=(14, 6))
plt.hist(X_train['Assembly Date'], bins=100, color='skyblue', edgecolor='black')
plt.title('Histogram of Assembly Dates', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45) 
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(X_train['Assembly Date'].describe())

Q1 = X_train['Assembly Date'].quantile(0.25)
Q3 = X_train['Assembly Date'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['Assembly Date'] < lower_bound) | (X_train['Assembly Date'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['Assembly Date'].describe())

lower_percentile = X_train['Assembly Date'].quantile(0.01)
upper_percentile = X_train['Assembly Date'].quantile(0.99)

print(f'Lower Bound (1st percentile): {lower_percentile}, Upper Bound (99th percentile): {upper_percentile}')

outliers = X_train[(X_train['Assembly Date'] < lower_percentile) | (X_train['Assembly Date'] > upper_percentile)]

print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
def clean_assembly_date_column(df, lower_percentile, upper_percentile, assembly_date_col='Assembly Date'):

    df[assembly_date_col] = df[assembly_date_col].apply(lambda x: x if (pd.notna(x) and lower_percentile <= x <= upper_percentile) else np.nan)
    
    return df

In [ ]:
X_train = clean_assembly_date_column(X_train, lower_percentile, upper_percentile, assembly_date_col='Assembly Date')
X_val = clean_assembly_date_column(X_val, lower_percentile, upper_percentile, assembly_date_col='Assembly Date')

In [ ]:
X_train['Assembly Date'].describe()

### C-2 Date

In [ ]:
plt.figure(figsize=(14, 6))
plt.hist(X_train['C-2 Date'], bins=100, color='skyblue', edgecolor='black')
plt.title('Histogram of C-2 Dates', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45) 
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(X_train['C-2 Date'].describe())

Q1 = X_train['C-2 Date'].quantile(0.25)
Q3 = X_train['C-2 Date'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['C-2 Date'] < lower_bound) | (X_train['C-2 Date'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['C-2 Date'].describe())

lower_percentile = X_train['C-2 Date'].quantile(0.01)
upper_percentile = X_train['C-2 Date'].quantile(0.99)

print(f'Lower Bound (1st percentile): {lower_percentile}, Upper Bound (99th percentile): {upper_percentile}')

outliers = X_train[(X_train['C-2 Date'] < lower_percentile) | (X_train['C-2 Date'] > upper_percentile)]

print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
def clean_c2_date_column(df, lower_percentile, upper_percentile, c2_date_col='C-2 Date'):
    
    df[c2_date_col] = df[c2_date_col].apply(lambda x: x if (pd.notna(x) and lower_percentile <= x <= upper_percentile) else np.nan)
    
    return df

In [ ]:
X_train = clean_c2_date_column(X_train, lower_percentile, upper_percentile, c2_date_col='C-2 Date')
X_val = clean_c2_date_column(X_val, lower_percentile, upper_percentile, c2_date_col='C-2 Date')

In [ ]:
X_train['C-2 Date'].describe()

### C-3 Date

In [ ]:
plt.figure(figsize=(14, 6))
plt.hist(X_train['C-3 Date'], bins=100, color='skyblue', edgecolor='black')
plt.title('Histogram of C-3 Dates', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45) 
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(X_train['C-3 Date'].describe())

Q1 = X_train['C-3 Date'].quantile(0.25)
Q3 = X_train['C-3 Date'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['C-3 Date'] < lower_bound) | (X_train['C-3 Date'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['C-3 Date'].describe())

lower_percentile = X_train['C-3 Date'].quantile(0.005)
upper_percentile = X_train['C-3 Date'].quantile(0.995)

print(f'Lower Bound (0.5th percentile): {lower_percentile}, Upper Bound (99.5th percentile): {upper_percentile}')

outliers = X_train[(X_train['C-3 Date'] < lower_percentile) | (X_train['C-3 Date'] > upper_percentile)]

print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
def clean_c3_date_column(df, lower_percentile, upper_percentile, c3_date_col='C-3 Date'):

    df[c3_date_col] = df[c3_date_col].apply(lambda x: x if (pd.notna(x) and lower_percentile <= x <= upper_percentile) else np.nan)
    
    return df

In [ ]:
X_train = clean_c3_date_column(X_train, lower_percentile, upper_percentile, c3_date_col='C-3 Date')
X_val = clean_c3_date_column(X_val, lower_percentile, upper_percentile, c3_date_col='C-3 Date')

In [ ]:
X_train['C-3 Date'].describe()

### First Hearing Date

In [ ]:
plt.figure(figsize=(14, 6))
plt.hist(X_train['First Hearing Date'], bins=100, color='skyblue', edgecolor='black')
plt.title('Histogram of First Hearing Date', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45) 
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print(X_train['First Hearing Date'].describe())

Q1 = X_train['First Hearing Date'].quantile(0.25)
Q3 = X_train['First Hearing Date'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(f'Lower Bound: {lower_bound}, Upper Bound: {upper_bound}')

outliers = X_train[(X_train['First Hearing Date'] < lower_bound) | (X_train['First Hearing Date'] > upper_bound)]
print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
print(X_train['First Hearing Date'].describe())

lower_percentile = X_train['First Hearing Date'].quantile(0.01)
upper_percentile = X_train['First Hearing Date'].quantile(0.99)

print(f'Lower Bound (1st percentile): {lower_percentile}, Upper Bound (99th percentile): {upper_percentile}')

outliers = X_train[(X_train['First Hearing Date'] < lower_percentile) | (X_train['First Hearing Date'] > upper_percentile)]

print(f'Number of Outliers: {outliers.shape[0]}')

In [ ]:
def clean_first_hearing_date_column(df, lower_percentile, upper_percentile, hearing_date_col='First Hearing Date'):
    df[hearing_date_col] = df[hearing_date_col].apply(
        lambda x: x if (pd.notna(x) and lower_percentile <= x <= upper_percentile) else np.nan
    )
    return df

In [ ]:
X_train = clean_first_hearing_date_column(X_train, lower_percentile, upper_percentile, hearing_date_col='First Hearing Date')
X_val = clean_first_hearing_date_column(X_val, lower_percentile, upper_percentile, hearing_date_col='First Hearing Date')

In [ ]:
X_train['First Hearing Date'].describe()

## Dropping 'Description' Variables

There are several variables in the dataset that contain both a 'code' and a 'description' representing the same information. To facilitate our analysis and reduce redundancy, we will store the descriptions in dictionaries, mapping each code to its corresponding description. Afterward, we will drop the description variables from the dataset to simplify the data structure and focus on the essential information for our analysis.

In [ ]:
industry_Code_train = X_train[['Industry Code', 'Industry Code Description']]
industry_Code_train = industry_Code_train.sort_values('Industry Code')
industry_Code_dict_train = industry_Code_train.set_index('Industry Code')['Industry Code Description'].dropna().to_dict()

industry_Code_dict_train

In [ ]:
WCIO_Cause_Injury = X_train[['WCIO Cause of Injury Code' , 'WCIO Cause of Injury Description']] 
WCIO_Cause_Injury = WCIO_Cause_Injury.sort_values('WCIO Cause of Injury Code')
WCIO_Cause_Injury_dict = WCIO_Cause_Injury.set_index('WCIO Cause of Injury Code')['WCIO Cause of Injury Description'].dropna().to_dict()

WCIO_Cause_Injury_dict

In [ ]:
WCIO_Nature_Injury = X_train[['WCIO Nature of Injury Code' , 'WCIO Nature of Injury Description']] 
WCIO_Nature_Injury = WCIO_Nature_Injury.sort_values('WCIO Nature of Injury Code')
WCIO_Nature_Injury_dict = WCIO_Nature_Injury.set_index('WCIO Nature of Injury Code')['WCIO Nature of Injury Description'].dropna().to_dict()

WCIO_Nature_Injury_dict

In [ ]:
WCIO_Part_of_Body = X_train[['WCIO Part Of Body Code' , 'WCIO Part Of Body Description']] 
WCIO_Part_of_Body = WCIO_Part_of_Body.sort_values('WCIO Part Of Body Code')
WCIO_Part_of_Body_dict = WCIO_Part_of_Body.set_index('WCIO Part Of Body Code')['WCIO Part Of Body Description'].dropna().to_dict()

WCIO_Part_of_Body_dict

In [ ]:
X_train = X_train.drop(['WCIO Part Of Body Description', 'WCIO Nature of Injury Description', 
                        'WCIO Cause of Injury Description', 'Industry Code Description'], axis=1)

X_val = X_val.drop(['WCIO Part Of Body Description', 'WCIO Nature of Injury Description', 
                    'WCIO Cause of Injury Description', 'Industry Code Description'], axis=1)

df_test = df_test.drop(['WCIO Part Of Body Description', 'WCIO Nature of Injury Description', 
                        'WCIO Cause of Injury Description', 'Industry Code Description'], axis=1)

## Missing Values

In [ ]:
X_train.isnull().mean()

### Accident Date

In [ ]:
print(f"Missing values in X_train (Accident Date): {X_train['Accident Date'].isnull().sum()}")
print(f"Missing values in X_val (Accident Date): {X_val['Accident Date'].isnull().sum()}")
print(f"Missing values in df_test (Accident Date): {df_test['Accident Date'].isnull().sum()}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.hist(X_train['Accident Date'], bins=100, color='skyblue', edgecolor='black')
plt.title('Histogram of Accident Dates', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.xticks(rotation=45) 
plt.grid(alpha=0.3)
plt.show()

In [ ]:
non_null_dates = X_train['Accident Date'].dropna()

mean_date = non_null_dates.mean()
median_date = non_null_dates.median()
mode_date = non_null_dates.mode()[0]

print(f"Mean Accident Date: {mean_date}")
print(f"Median Accident Date: {median_date}")
print(f"Mode Accident Date: {mode_date}")

Due to the skewness observed in the distribution of the Accident Date column, we decided to impute the missing values with the median.

In [ ]:
def impute_accident_year(X_train, X_val, df_test):

    accident_year_mean = non_null_dates.median()

    X_train['Accident Date'] = X_train['Accident Date'].fillna(accident_year_mean)
    X_val['Accident Date'] = X_val['Accident Date'].fillna(accident_year_mean)
    df_test['Accident Date'] = df_test['Accident Date'].fillna(accident_year_mean)

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_accident_year(X_train, X_val, df_test)

In [ ]:
print(f"Missing values in X_train (Accident Date): {X_train['Accident Date'].isnull().sum()}")
print(f"Missing values in X_val (Accident Date): {X_val['Accident Date'].isnull().sum()}")
print(f"Missing values in df_test (Accident Date): {df_test['Accident Date'].isnull().sum()}")

### Age at Injury

In [ ]:
print(f"Missing values in X_train: {X_train['Age at Injury'].isnull().sum()}")
print(f"Missing values in X_val: {X_val['Age at Injury'].isnull().sum()}")
print(f"Missing values in df_test: {df_test['Age at Injury'].isnull().sum()}")

In [ ]:
age_data = X_train['Age at Injury'].dropna()

plt.figure(figsize=(10, 6))
plt.hist(age_data, bins=30, color='skyblue', edgecolor='black')

plt.title('Age at Injury Distribution in X_train')
plt.xlabel('Age at Injury')
plt.ylabel('Frequency')

plt.show()

In [ ]:
mean_age = age_data.mean()
median_age = age_data.median()

print(f"Mean Age at Injury: {mean_age:.2f}")
print(f"Median Age at Injury: {median_age:.2f}")

In this case, mean and median imputation is very similar. We are going to use the median.

In [ ]:
def impute_age_at_injury(X_train, X_val, df_test):

    imputer = SimpleImputer(strategy='median')

    X_train['Age at Injury'] = imputer.fit_transform(X_train[['Age at Injury']])
    X_val['Age at Injury'] = imputer.transform(X_val[['Age at Injury']])
    df_test['Age at Injury'] = imputer.transform(df_test[['Age at Injury']])

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_age_at_injury(X_train, X_val, df_test)

In [ ]:
print(f"Missing values in X_train: {X_train['Age at Injury'].isnull().sum()}")
print(f"Missing values in X_val: {X_val['Age at Injury'].isnull().sum()}")
print(f"Missing values in df_test: {df_test['Age at Injury'].isnull().sum()}")

### Alternative Dispute Resolution

In [ ]:
X_train['Alternative Dispute Resolution'].value_counts()

In [ ]:
X_val['Alternative Dispute Resolution'].value_counts()

In [ ]:
df_test['Alternative Dispute Resolution'].value_counts()

We will impute these Unknowns with the mode.

In [ ]:
def impute_adr_mode(X_train, X_val, df_test):

    ADR_mode = X_train['Alternative Dispute Resolution'].mode()[0]
    
    X_train['Alternative Dispute Resolution'] = X_train['Alternative Dispute Resolution'].replace('U', ADR_mode)
    X_val['Alternative Dispute Resolution'] = X_val['Alternative Dispute Resolution'].replace('U', ADR_mode)
    df_test['Alternative Dispute Resolution'] = df_test['Alternative Dispute Resolution'].replace('U', ADR_mode)

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_adr_mode(X_train, X_val, df_test)

In [ ]:
X_train['Alternative Dispute Resolution'].value_counts()

### Assembly Date

In [ ]:
print(f"Null values in X_train 'Assembly Date': {X_train['Assembly Date'].isna().sum()}")
print(f"Null values in X_val 'Assembly Date': {X_val['Assembly Date'].isna().sum()}")
print(f"Null values in df_test 'Assembly Date': {df_test['Assembly Date'].isna().sum()}")

Since there are no missing values in df_test, we decided to drop the rows with missing values in both X_train and X_val. 

In [ ]:
X_train = X_train.dropna(subset=['Assembly Date'])
X_val = X_val.dropna(subset=['Assembly Date'])

y_train = y_train.loc[X_train.index]
y_val = y_val.loc[X_val.index]

In [ ]:
print(f"Null values in X_train 'Assembly Date': {X_train['Assembly Date'].isna().sum()}")
print(f"Null values in X_val 'Assembly Date': {X_val['Assembly Date'].isna().sum()}")
print(f"Null values in df_test 'Assembly Date': {df_test['Assembly Date'].isna().sum()}")

### C-2 Date

In [ ]:
print(f"Null values in X_train 'C-2 Date': {X_train['C-2 Date'].isna().sum()}")
print(f"Null values in X_val 'C-2 Date': {X_val['C-2 Date'].isna().sum()}")
print(f"Null values in df_test 'C-2 Date': {df_test['C-2 Date'].isna().sum()}")

In [ ]:
def fill_c2_date_with_bfill(X_train, X_val, df_test):

    X_train_original_order = X_train.index
    X_val_original_order = X_val.index
    df_test_original_order = df_test.index

    X_train_sorted = X_train.sort_values(by='Accident Date', ascending=True)
    X_val_sorted = X_val.sort_values(by='Accident Date', ascending=True)
    df_test_sorted = df_test.sort_values(by='Accident Date', ascending=True)
    
    X_train_sorted['C-2 Date'] = X_train_sorted['C-2 Date'].bfill()
    X_val_sorted['C-2 Date'] = X_val_sorted['C-2 Date'].bfill()
    df_test_sorted['C-2 Date'] = df_test_sorted['C-2 Date'].bfill()
    
    X_train_sorted = X_train_sorted.loc[X_train_original_order]
    X_val_sorted = X_val_sorted.loc[X_val_original_order]
    df_test_sorted = df_test_sorted.loc[df_test_original_order]
    
    return X_train_sorted, X_val_sorted, df_test_sorted

In [ ]:
X_train, X_val, df_test = fill_c2_date_with_bfill(X_train, X_val, df_test)

Here, we first ordered the data by 'Accident Date' to ensure that the 'C-2 Date' values are imputed in a chronological order. In this way we maintain the temporal relationship between the accident and the corresponding 'C-2 Date', which allows us to use the backfill method more effectively, filling missing values with the next available valid entry in the correct sequence.

In [ ]:
print(f"Null values in X_train 'C-2 Date': {X_train['C-2 Date'].isna().sum()}")
print(f"Null values in X_val 'C-2 Date': {X_val['C-2 Date'].isna().sum()}")
print(f"Null values in df_test 'C-2 Date': {df_test['C-2 Date'].isna().sum()}")

### C-3 Date

In [ ]:
print(f"Null values in X_train 'C-3 Date': {X_train['C-3 Date'].isna().sum()}")
print(f"Null values in X_val 'C-3 Date': {X_val['C-3 Date'].isna().sum()}")
print(f"Null values in df_test 'C-3 Date': {df_test['C-3 Date'].isna().sum()}")

We treated the null values in the 'C-3 Date' column as cases where the 'C-3' form was not received. To handle this, we created a new binary variable, 'C-3 Received,' where a value of 0 indicates that the 'C-3 Date' is null (meaning the form was not received), and a value of 1 indicates that the 'C-3 Date' exists (meaning the form was received).

In [ ]:
X_train['C-3 Received'] = X_train['C-3 Date'].apply(lambda x: 0 if pd.isnull(x) else 1)
X_train = X_train.drop(columns=['C-3 Date'])

In [ ]:
X_val['C-3 Received'] = X_val['C-3 Date'].apply(lambda x: 0 if pd.isnull(x) else 1)
X_val = X_val.drop(columns=['C-3 Date'])

In [ ]:
df_test['C-3 Received'] = df_test['C-3 Date'].apply(lambda x: 0 if pd.isnull(x) else 1)
df_test = df_test.drop(columns=['C-3 Date'])

### First Hearing Date

In [ ]:
print(f"Null values in X_train 'First Hearing Date': {X_train['First Hearing Date'].isna().sum()}")
print(f"Null values in X_val 'First Hearing Date': {X_val['First Hearing Date'].isna().sum()}")
print(f"Null values in df_test 'First Hearing Date': {df_test['First Hearing Date'].isna().sum()}")

Similarly, for the 'First Hearing Date' column, we treated the null values as instances where the first hearing had not yet occurred. To address this, we created a binary variable called 'First Hearing Happened.' A value of 0 in this column indicates that the 'First Hearing Date' is null (meaning the hearing has not occurred), and a value of 1 indicates that the date is present (meaning the hearing has taken place). 

In [ ]:
X_train['First Hearing Happened'] = X_train['First Hearing Date'].apply(lambda x: 0 if pd.isnull(x) else 1)
X_train = X_train.drop(columns=['First Hearing Date'])

In [ ]:
X_val['First Hearing Happened'] = X_val['First Hearing Date'].apply(lambda x: 0 if pd.isnull(x) else 1)
X_val = X_val.drop(columns=['First Hearing Date'])

In [ ]:
df_test['First Hearing Happened'] = df_test['First Hearing Date'].apply(lambda x: 0 if pd.isnull(x) else 1)
df_test = df_test.drop(columns=['First Hearing Date'])

### Attorney/Representative

In [ ]:
X_train['Attorney/Representative'].value_counts()

In [ ]:
X_val['Attorney/Representative'].value_counts()

In [ ]:
df_test['Attorney/Representative'].value_counts()

Everything looks fine!

### Carrier Type

In [ ]:
print(f"Missing values in X_train: {X_train['Carrier Type'].isnull().sum()}")
print(f"Missing values in X_val: {X_val['Carrier Type'].isnull().sum()}")
print(f"Missing values in df_test: {df_test['Carrier Type'].isnull().sum()}")

### Carrier Name

In [ ]:
print(f"Missing values in X_train: {X_train['Carrier Name'].isnull().sum()}")
print(f"Missing values in X_val: {X_val['Carrier Name'].isnull().sum()}")
print(f"Missing values in df_test: {df_test['Carrier Name'].isnull().sum()}")

### Industry Code

In [ ]:
print(f"Missing values in X_train: {X_train['Industry Code'].isnull().sum()}")
print(f"Missing values in X_val: {X_val['Industry Code'].isnull().sum()}")
print(f"Missing values in df_test: {df_test['Industry Code'].isnull().sum()}")

In [ ]:
def impute_industry_code(X_train, X_val, df_test):

    industry_code_mode = X_train.groupby('Carrier Type')['Industry Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )

    def fill_industry_code(row):
        if pd.notnull(row['Industry Code']):
            return row['Industry Code']
        elif pd.notnull(row['Carrier Type']):
            return industry_code_mode.get(row['Carrier Type'], row['Industry Code'])
        else:
            return row['Industry Code']

    X_train['Industry Code'] = X_train.apply(fill_industry_code, axis=1)
    X_val['Industry Code'] = X_val.apply(fill_industry_code, axis=1)
    df_test['Industry Code'] = df_test.apply(fill_industry_code, axis=1)

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_industry_code(X_train, X_val, df_test)

We imputed missing values in the 'Industry Code' column based on the most frequent 'Industry Code' for each 'Carrier Type' in the training data. For each row, if the 'Industry Code' is missing and the 'Carrier Type' is available, we replaced the missing value with the most common 'Industry Code' associated with that 'Carrier Type'. If both values were missing, the 'Industry Code' was left as NaN.

In [ ]:
print(f"Missing values in X_train: {X_train['Carrier Type'].isnull().sum()}")
print(f"Missing values in X_val: {X_val['Carrier Type'].isnull().sum()}")
print(f"Missing values in df_test: {df_test['Carrier Type'].isnull().sum()}")

### Average Weekly Wage

In [ ]:
print(f"Missing values in X_train: {X_train['Average Weekly Wage'].isnull().sum()}")
print(f"Missing values in X_val: {X_val['Average Weekly Wage'].isnull().sum()}")
print(f"Missing values in df_test: {df_test['Average Weekly Wage'].isnull().sum()}")

In [ ]:
def impute_aww(X_train, X_val, df_test):

    X_train_no_nulls = X_train[X_train['Average Weekly Wage'].notnull()]
    

    wage_mean = X_train_no_nulls.groupby(['Industry Code', 'Carrier Type'])['Average Weekly Wage'].mean()
    wage_mean_industry = X_train_no_nulls.groupby('Industry Code')['Average Weekly Wage'].mean()
    wage_mean_carrier = X_train_no_nulls.groupby('Carrier Type')['Average Weekly Wage'].mean()
    overall_mean = X_train_no_nulls['Average Weekly Wage'].mean()


    def fill_wage(row):
        if pd.notnull(row['Average Weekly Wage']):
            return row['Average Weekly Wage']
        if pd.notnull(row['Industry Code']) and pd.notnull(row['Carrier Type']):
            return wage_mean.get((row['Industry Code'], row['Carrier Type']), overall_mean)
        elif pd.notnull(row['Industry Code']):
            return wage_mean_industry.get(row['Industry Code'], overall_mean)
        elif pd.notnull(row['Carrier Type']):
            return wage_mean_carrier.get(row['Carrier Type'], overall_mean)
        else:
            return overall_mean


    X_train['Average Weekly Wage'] = X_train.apply(fill_wage, axis=1)
    X_val['Average Weekly Wage'] = X_val.apply(fill_wage, axis=1)
    df_test['Average Weekly Wage'] = df_test.apply(fill_wage, axis=1)

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_aww(X_train, X_val, df_test)

First, we calculated the mean 'Average Weekly Wage' for combinations of 'Industry Code' and 'Carrier Type', as well as separately for each 'Industry Code' and 'Carrier Type'. If both 'Industry Code' and 'Carrier Type' were available, we used their combined mean. If only one of these variables was present, we used the corresponding mean for that variable. If neither was available, we used the overall mean 'Average Weekly Wage' from the entire dataset.

In [ ]:
print(f"Null values in X_train 'Average Weekly Wage': {X_train['Average Weekly Wage'].isna().sum()}")
print(f"Null values in X_val 'Average Weekly Wage': {X_val['Average Weekly Wage'].isna().sum()}")
print(f"Null values in df_test 'Average Weekly Wage': {df_test['Average Weekly Wage'].isna().sum()}")

### Birth Year

In [ ]:
print(f"Null values in X_train 'Birth Year': {X_train['Birth Year'].isna().sum()}")
print(f"Null values in X_val 'Birth Year': {X_val['Birth Year'].isna().sum()}")
print(f"Null values in df_test 'Birth Year': {df_test['Birth Year'].isna().sum()}")

In [ ]:
def impute_birth_year(X_train, X_val, df_test):
    def calculate_birth_year(row):
        if pd.notnull(row['Birth Year']):
            return row['Birth Year']
        if pd.notnull(row['Accident Date']) and pd.notnull(row['Age at Injury']):
            accident_year = row['Accident Date'].year  # Extract the year from the datetime
            return accident_year - row['Age at Injury']
        return row['Birth Year']
    
    X_train['Birth Year'] = X_train.apply(calculate_birth_year, axis=1)
    X_val['Birth Year'] = X_val.apply(calculate_birth_year, axis=1)
    df_test['Birth Year'] = df_test.apply(calculate_birth_year, axis=1)

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_birth_year(X_train, X_val, df_test)

We imputed missing values in the 'Birth Year' column by calculating it based on the 'Age at Injury' and 'Accident Date'. 

In [ ]:
print(f"Null values in X_train 'Birth Year': {X_train['Birth Year'].isna().sum()}")
print(f"Null values in X_val 'Birth Year': {X_val['Birth Year'].isna().sum()}")
print(f"Null values in df_test 'Birth Year': {df_test['Birth Year'].isna().sum()}")

### County of Injury

In [ ]:
print(f"Null values in X_train 'County of Injury': {X_train['County of Injury'].isna().sum()}")
print(f"Null values in X_val 'County of Injury': {X_val['County of Injury'].isna().sum()}")
print(f"Null values in df_test 'County of Injury': {df_test['County of Injury'].isna().sum()}")

### COVID-19 Indicator

In [ ]:
print(f"Null values in X_train 'COVID-19 Indicator': {X_train['COVID-19 Indicator'].isna().sum()}")
print(f"Null values in X_val 'COVID-19 Indicator': {X_val['COVID-19 Indicator'].isna().sum()}")
print(f"Null values in df_test 'COVID-19 Indicator': {df_test['COVID-19 Indicator'].isna().sum()}")

In [ ]:
X_train['COVID-19 Indicator'].value_counts()

In [ ]:
X_val['COVID-19 Indicator'].value_counts()

In [ ]:
df_test['COVID-19 Indicator'].value_counts()

### District Name

In [ ]:
print(f"Null values in X_train 'District Name': {X_train['District Name'].isna().sum()}")
print(f"Null values in X_val 'District Name': {X_val['District Name'].isna().sum()}")
print(f"Null values in df_test 'District Name': {df_test['District Name'].isna().sum()}")

In [ ]:
X_train['District Name'].value_counts()

In [ ]:
X_val['District Name'].value_counts()

In [ ]:
df_test['District Name'].value_counts()

### Gender

In [ ]:
print(f"Null values in X_train 'Gender': {X_train['Gender'].isna().sum()}")
print(f"Null values in X_val 'Gender': {X_val['Gender'].isna().sum()}")
print(f"Null values in df_test 'Gender': {df_test['Gender'].isna().sum()}")

In [ ]:
X_train['Gender'].value_counts()

In [ ]:
X_val['Gender'].value_counts()

In [ ]:
df_test['Gender'].value_counts()

### IME-4 Count

In [ ]:
print(f"Null values in X_train 'IME-4 Count': {X_train['IME-4 Count'].isna().sum()}")
print(f"Null values in X_val 'IME-4 Count': {X_val['IME-4 Count'].isna().sum()}")
print(f"Null values in df_test 'IME-4 Count': {df_test['IME-4 Count'].isna().sum()}")

In [ ]:
X_train['IME-4 Count'].value_counts()

In [ ]:
X_val['IME-4 Count'].value_counts()

In [ ]:
df_test['IME-4 Count'].value_counts()

The Count variable for IME-4 represents the number of IME-4 forms received for every claim. The IME-4 means an Independent Examiner's Report of Independent Medical Examination. Because the count of IME-4 forms has no zeros and there is a distribution value starting with 1, we suspect that the missing values likely correspond to people who did not visit a doctor for an examination and, therefore, should be regarded as having a count of 0 for the IME-4 forms. 

In [ ]:
X_train['IME-4 Count'] = X_train['IME-4 Count'].apply(lambda x: 0 if pd.isna(x) else x)
X_val['IME-4 Count'] = X_val['IME-4 Count'].apply(lambda x: 0 if pd.isna(x) else x)
df_test['IME-4 Count'] = df_test['IME-4 Count'].apply(lambda x: 0 if pd.isna(x) else x)

In [ ]:
print(f"Null values in X_train 'IME-4 Count': {X_train['IME-4 Count'].isna().sum()}")
print(f"Null values in X_val 'IME-4 Count': {X_val['IME-4 Count'].isna().sum()}")
print(f"Null values in df_test 'IME-4 Count': {df_test['IME-4 Count'].isna().sum()}")

### Medical Fee Region

In [ ]:
print(f"Null values in X_train 'Medical Fee Region': {X_train['Medical Fee Region'].isna().sum()}")
print(f"Null values in X_val 'Medical Fee Region': {X_val['Medical Fee Region'].isna().sum()}")
print(f"Null values in df_test 'Medical Fee Region': {df_test['Medical Fee Region'].isna().sum()}")

In [ ]:
X_train['Medical Fee Region'].value_counts()

In [ ]:
X_val['Medical Fee Region'].value_counts()

In [ ]:
df_test['Medical Fee Region'].value_counts()

There are no missing values, but there are some 'UK' values. We will fill the 'UK' values in the 'Medical Fee Region' based on the most common Medical Fee Region for each County of Injury, as these variables are intuitively related.

In [ ]:
def impute_medical_fee_region(X_train, X_val, df_test):

    X_train_without_UK = X_train[X_train['Medical Fee Region'] != 'UK']
    most_common_fee_region = X_train_without_UK.groupby('County of Injury')['Medical Fee Region'].agg(lambda x: x.mode()[0])

    def fill_fee_region(row):
        if row['Medical Fee Region'] == 'UK' and pd.notnull(row['County of Injury']):
            return most_common_fee_region.get(row['County of Injury'], row['Medical Fee Region'])
        return row['Medical Fee Region']
    
    X_train['Medical Fee Region'] = X_train.apply(fill_fee_region, axis=1)
    X_val['Medical Fee Region'] = X_val.apply(fill_fee_region, axis=1)
    df_test['Medical Fee Region'] = df_test.apply(fill_fee_region, axis=1)

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_medical_fee_region(X_train, X_val, df_test)

In [ ]:
X_train['Medical Fee Region'].value_counts()

In [ ]:
X_val['Medical Fee Region'].value_counts()

In [ ]:
df_test['Medical Fee Region'].value_counts()

### OIICS Nature of Injury Description

In [ ]:
print(f"Null values in X_train 'OIICS Nature of Injury Description': {X_train['OIICS Nature of Injury Description'].isna().sum()}")
print(f"Null values in X_val 'OIICS Nature of Injury Description': {X_val['OIICS Nature of Injury Description'].isna().sum()}")
print(f"Null values in df_test 'OIICS Nature of Injury Description': {df_test['OIICS Nature of Injury Description'].isna().sum()}")

In [ ]:
X_train.drop(['OIICS Nature of Injury Description'], axis=1, inplace=True)
X_val.drop(['OIICS Nature of Injury Description'], axis=1, inplace=True)
df_test.drop(['OIICS Nature of Injury Description'], axis=1, inplace=True)

### WCIO Cause of Injury Code

In [ ]:
print(f"Null values in X_train 'WCIO Cause of Injury Code': {X_train['WCIO Cause of Injury Code'].isna().sum()}")
print(f"Null values in X_val 'WCIO Cause of Injury Code': {X_val['WCIO Cause of Injury Code'].isna().sum()}")
print(f"Null values in df_test 'WCIO Cause of Injury Code': {df_test['WCIO Cause of Injury Code'].isna().sum()}")

To address these missing values, we will use group-based imputation based on the 'Carrier Type' and 'Injury Code'.

In [ ]:
def impute_wcicause(X_train, X_val, df_test):
    overall_mode = X_train['WCIO Cause of Injury Code'].mode()[0]
    wcio_cause_mode = X_train.groupby(['Carrier Type', 'Industry Code'])['WCIO Cause of Injury Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    wcio_cause_mode_carrier = X_train.groupby('Carrier Type')['WCIO Cause of Injury Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    wcio_cause_mode_industry = X_train.groupby('Industry Code')['WCIO Cause of Injury Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)

    def fill_wcicause(row):
        if pd.notnull(row['WCIO Cause of Injury Code']):
            return row['WCIO Cause of Injury Code']
        if pd.notnull(row['Carrier Type']) and pd.notnull(row['Industry Code']):
            return wcio_cause_mode.get((row['Carrier Type'], row['Industry Code']), overall_mode)
        elif pd.notnull(row['Carrier Type']):
            return wcio_cause_mode_carrier.get(row['Carrier Type'], overall_mode)
        elif pd.notnull(row['Industry Code']):
            return wcio_cause_mode_industry.get(row['Industry Code'], overall_mode)
        else:
            return overall_mode

    X_train['WCIO Cause of Injury Code'] = X_train.apply(fill_wcicause, axis=1)
    X_val['WCIO Cause of Injury Code'] = X_val.apply(fill_wcicause, axis=1)
    df_test['WCIO Cause of Injury Code'] = df_test.apply(fill_wcicause, axis=1)

    X_train['WCIO Cause of Injury Code'] = X_train['WCIO Cause of Injury Code'].astype('float64').round().astype('int64')
    X_val['WCIO Cause of Injury Code'] = X_val['WCIO Cause of Injury Code'].astype('float64').round().astype('int64')
    df_test['WCIO Cause of Injury Code'] = df_test['WCIO Cause of Injury Code'].astype('float64').round().astype('int64')

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_wcicause(X_train, X_val, df_test)

In [ ]:
print(f"Null values in X_train 'WCIO Cause of Injury Code': {X_train['WCIO Cause of Injury Code'].isna().sum()}")
print(f"Null values in X_val 'WCIO Cause of Injury Code': {X_val['WCIO Cause of Injury Code'].isna().sum()}")
print(f"Null values in df_test 'WCIO Cause of Injury Code': {df_test['WCIO Cause of Injury Code'].isna().sum()}")

### WCIO Nature of Injury Code

In [ ]:
print(f"Null values in X_train 'WCIO Nature of Injury Code': {X_train['WCIO Nature of Injury Code'].isna().sum()}")
print(f"Null values in X_val 'WCIO Nature of Injury Code': {X_val['WCIO Nature of Injury Code'].isna().sum()}")
print(f"Null values in df_test 'WCIO Nature of Injury Code': {df_test['WCIO Nature of Injury Code'].isna().sum()}")

We will use group-based imputation based on 'Injury Code' and 'WCIO Cause of Injury Code' to fill the missing values.

In [ ]:
def impute_wcionature(X_train, X_val, df_test):
    overall_mode = X_train['WCIO Nature of Injury Code'].mode()[0]
    
    wcio_nature_mode = X_train.groupby(['Industry Code', 'WCIO Cause of Injury Code'])['WCIO Nature of Injury Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    wcio_nature_mode_industry = X_train.groupby('Industry Code')['WCIO Nature of Injury Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    wcio_nature_mode_cause = X_train.groupby('WCIO Cause of Injury Code')['WCIO Nature of Injury Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)

    def fill_wcionature(row):
        if pd.notnull(row['WCIO Nature of Injury Code']):
            return row['WCIO Nature of Injury Code']
        if pd.notnull(row['Industry Code']) and pd.notnull(row['WCIO Cause of Injury Code']):
            return wcio_nature_mode.get((row['Industry Code'], row['WCIO Cause of Injury Code']), overall_mode)
        elif pd.notnull(row['Industry Code']):
            return wcio_nature_mode_industry.get(row['Industry Code'], overall_mode)
        elif pd.notnull(row['WCIO Cause of Injury Code']):
            return wcio_nature_mode_cause.get(row['WCIO Cause of Injury Code'], overall_mode)
        else:
            return overall_mode

    X_train['WCIO Nature of Injury Code'] = X_train.apply(fill_wcionature, axis=1)
    X_val['WCIO Nature of Injury Code'] = X_val.apply(fill_wcionature, axis=1)
    df_test['WCIO Nature of Injury Code'] = df_test.apply(fill_wcionature, axis=1)

    X_train['WCIO Nature of Injury Code'] = X_train['WCIO Nature of Injury Code'].astype('float64').round().astype('int64')
    X_val['WCIO Nature of Injury Code'] = X_val['WCIO Nature of Injury Code'].astype('float64').round().astype('int64')
    df_test['WCIO Nature of Injury Code'] = df_test['WCIO Nature of Injury Code'].astype('float64').round().astype('int64')

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_wcionature(X_train, X_val, df_test)

In [ ]:
print(f"Null values in X_train 'WCIO Nature of Injury Code': {X_train['WCIO Nature of Injury Code'].isna().sum()}")
print(f"Null values in X_val 'WCIO Nature of Injury Code': {X_val['WCIO Nature of Injury Code'].isna().sum()}")
print(f"Null values in df_test 'WCIO Nature of Injury Code': {df_test['WCIO Nature of Injury Code'].isna().sum()}")

### WCIO Part Of Body Code

In [ ]:
print(f"Null values in X_train 'WCIO Part Of Body Code': {X_train['WCIO Part Of Body Code'].isna().sum()}")
print(f"Null values in X_val 'WCIO Part Of Body Code': {X_val['WCIO Part Of Body Code'].isna().sum()}")
print(f"Null values in df_test 'WCIO Part Of Body Code': {df_test['WCIO Part Of Body Code'].isna().sum()}")

We will use group-based imputation based on 'WCIO Cause of Injury Code' and 'WCIO Nature of Injury Code' to fill the missing values.

In [ ]:
def impute_wciopart(X_train, X_val, df_test):
    overall_mode = X_train['WCIO Part Of Body Code'].mode()[0]
    
    wcio_part_mode = X_train.groupby(['WCIO Cause of Injury Code', 'WCIO Nature of Injury Code'])['WCIO Part Of Body Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    wcio_part_mode_cause = X_train.groupby('WCIO Cause of Injury Code')['WCIO Part Of Body Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    wcio_part_mode_nature = X_train.groupby('WCIO Nature of Injury Code')['WCIO Part Of Body Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)

    def fill_wciopart(row):
        if pd.notnull(row['WCIO Part Of Body Code']):
            return row['WCIO Part Of Body Code']
        if pd.notnull(row['WCIO Cause of Injury Code']) and pd.notnull(row['WCIO Nature of Injury Code']):
            return wcio_part_mode.get((row['WCIO Cause of Injury Code'], row['WCIO Nature of Injury Code']), overall_mode)
        elif pd.notnull(row['WCIO Cause of Injury Code']):
            return wcio_part_mode_cause.get(row['WCIO Cause of Injury Code'], overall_mode)
        elif pd.notnull(row['WCIO Nature of Injury Code']):
            return wcio_part_mode_nature.get(row['WCIO Nature of Injury Code'], overall_mode)
        else:
            return overall_mode

    X_train['WCIO Part Of Body Code'] = X_train.apply(fill_wciopart, axis=1)
    X_val['WCIO Part Of Body Code'] = X_val.apply(fill_wciopart, axis=1)
    df_test['WCIO Part Of Body Code'] = df_test.apply(fill_wciopart, axis=1)

    X_train['WCIO Part Of Body Code'] = X_train['WCIO Part Of Body Code'].astype('float64').round().astype('int64')
    X_val['WCIO Part Of Body Code'] = X_val['WCIO Part Of Body Code'].astype('float64').round().astype('int64')
    df_test['WCIO Part Of Body Code'] = df_test['WCIO Part Of Body Code'].astype('float64').round().astype('int64')

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_wciopart(X_train, X_val, df_test)

In [ ]:
print(f"Null values in X_train 'WCIO Part Of Body Code': {X_train['WCIO Part Of Body Code'].isna().sum()}")
print(f"Null values in X_val 'WCIO Part Of Body Code': {X_val['WCIO Part Of Body Code'].isna().sum()}")
print(f"Null values in df_test 'WCIO Part Of Body Code': {df_test['WCIO Part Of Body Code'].isna().sum()}")

### Zip Code

In [ ]:
X_train['Zip Code'].value_counts()

We noticed that some zip code values were stored as floats, while others were integers. This inconsistency led to multiple categories being created for the same zip code. To resolve this, we wrote a function to standardize the zip codes.

In [ ]:
def convert_zip_code(x):
    if pd.notnull(x) and isinstance(x, (int, float, str)):
        try:
            return str(int(float(x)))
        except ValueError:
            return str(x)
    return x

In [ ]:
X_train['Zip Code'] = X_train['Zip Code'].apply(convert_zip_code)
X_val['Zip Code'] = X_val['Zip Code'].apply(convert_zip_code)
df_test['Zip Code'] = df_test['Zip Code'].apply(convert_zip_code)

In [ ]:
X_train['Zip Code'].value_counts()

In [ ]:
print(f"Null values in X_train 'Zip Code': {X_train['Zip Code'].isna().sum()}")
print(f"Null values in X_val 'Zip Code': {X_val['Zip Code'].isna().sum()}")
print(f"Null values in df_test 'Zip Code': {df_test['Zip Code'].isna().sum()}")

To fill these missing values, we will use a group-based imputation based on 'County of Injury' and 'Medical Fee Region'.

In [ ]:
def impute_zip(X_train, X_val, df_test):

    overall_mode = X_train['Zip Code'].mode()[0]
    
    zip_mode = X_train.groupby(['County of Injury', 'Medical Fee Region'])['Zip Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    zip_mode_county = X_train.groupby('County of Injury')['Zip Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)
    
    zip_mode_region = X_train.groupby('Medical Fee Region')['Zip Code'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else overall_mode)

    def fill_zip(row):
        if pd.notnull(row['Zip Code']):
            return row['Zip Code']
        if pd.notnull(row['County of Injury']) and pd.notnull(row['Medical Fee Region']):
            return zip_mode.get((row['County of Injury'], row['Medical Fee Region']), overall_mode)
        elif pd.notnull(row['County of Injury']):
            return zip_mode_county.get(row['County of Injury'], overall_mode)
        elif pd.notnull(row['Medical Fee Region']):
            return zip_mode_region.get(row['Medical Fee Region'], overall_mode)
        else:
            return overall_mode

    X_train['Zip Code'] = X_train.apply(fill_zip, axis=1)
    X_val['Zip Code'] = X_val.apply(fill_zip, axis=1)
    df_test['Zip Code'] = df_test.apply(fill_zip, axis=1)

    return X_train, X_val, df_test

In [ ]:
X_train, X_val, df_test = impute_zip(X_train, X_val, df_test)

In [ ]:
print(f"Null values in X_train 'Zip Code': {X_train['Zip Code'].isna().sum()}")
print(f"Null values in X_val 'Zip Code': {X_val['Zip Code'].isna().sum()}")
print(f"Null values in df_test 'Zip Code': {df_test['Zip Code'].isna().sum()}")

### Agreement Reached

In [ ]:
print(f"Null values in X_train 'Agreement Reached': {X_train['Agreement Reached'].isna().sum()}")
print(f"Null values in X_val 'Agreement Reached': {X_val['Agreement Reached'].isna().sum()}")

In [ ]:
X_train['Agreement Reached'].value_counts()

In [ ]:
X_val['Agreement Reached'].value_counts()

All ok!

### WCB Decision

In [ ]:
print(f"Null values in X_train 'WCB Decision': {X_train['WCB Decision'].isna().sum()}")
print(f"Null values in X_val 'WCB Decision': {X_val['WCB Decision'].isna().sum()}")

In [ ]:
X_train['WCB Decision'].value_counts()

In [ ]:
X_val['WCB Decision'].value_counts()

Since the 'WCB Decision' column contains only a single value ("Not Work Related") across all rows, it does not provide any useful information for analysis. Therefore, we will drop this column from the dataset.

In [ ]:
X_train.drop(columns=['WCB Decision'], inplace=True)
X_val.drop(columns=['WCB Decision'], inplace=True)

### Number of Dependents

In [ ]:
print(f"Null values in X_train 'Number of Dependents': {X_train['Number of Dependents'].isna().sum()}")
print(f"Null values in X_val 'Number of Dependents': {X_val['Number of Dependents'].isna().sum()}")
print(f"Null values in df_test 'Number of Dependents': {df_test['Number of Dependents'].isna().sum()}")

In [ ]:
X_train['Number of Dependents'].value_counts()

In [ ]:
X_val['Number of Dependents'].value_counts()

In [ ]:
df_test['Number of Dependents'].value_counts()

## Extra features

### Injury Day of Week

Extract weekday from Accident Date.

In [ ]:
for df in [X_train, X_val, df_test]:
    df['Accident Date'] = pd.to_datetime(df['Accident Date'])
    df['Injury Day of Week'] = df['Accident Date'].dt.day_name()

In [ ]:
X_train[['Accident Date', 'Injury Day of Week']].head()

### Time to Assembly

Difference in days between Accident Date and Assembly Date.

In [ ]:
for df in [X_train, X_val, df_test]:
    df['Time to Assembly'] = (df['Assembly Date'] - df['Accident Date']).dt.days

In [ ]:
X_train[['Accident Date', 'Assembly Date', 'Time to Assembly']].head()

**Fixing negative values:**

In [ ]:
for df in [X_train, X_val, df_test]:
    negative_count = (df['Time to Assembly'] < 0).sum()
    print(f"Negative 'Time to Assembly' count in dataset: {negative_count}")

In [ ]:
def replace_negatives_with_avg(dfs, column_name, reference_df):

    avg_value = int(reference_df[column_name].loc[reference_df[column_name] >= 0].mean())

    for df in dfs:
        df[column_name] = df[column_name].apply(lambda x: avg_value if x < 0 else x)
    
    return dfs

In [ ]:
X_train, X_val, df_test = replace_negatives_with_avg(
    [X_train, X_val, df_test], 
    column_name='Time to Assembly', 
    reference_df=X_train
)

In [ ]:
for df in [X_train, X_val, df_test]:
    negative_count = (df['Time to Assembly'] < 0).sum()
    print(f"Negative 'Time to Assembly' count in dataset: {negative_count}")

### Time to C-2

Difference in days between Accident Date and C-2 Date.

In [ ]:
for df in [X_train, X_val, df_test]:
    df['Time to C-2'] = (df['C-2 Date'] - df['Accident Date']).dt.days

In [ ]:
X_train[['Accident Date', 'C-2 Date', 'Time to C-2']].head()

**Fixing negative values:**

In [ ]:
for df in [X_train, X_val, df_test]:
    negative_count = (df['Time to C-2'] < 0).sum()
    print(f"Negative 'Time to C-2' count in dataset: {negative_count}")

In [ ]:
X_train, X_val, df_test = replace_negatives_with_avg(
    [X_train, X_val, df_test], 
    column_name='Time to C-2', 
    reference_df=X_train
)

In [ ]:
for df in [X_train, X_val, df_test]:
    negative_count = (df['Time to C-2'] < 0).sum()
    print(f"Negative 'Time to C-2' count in dataset: {negative_count}")

### Wage/Dependents Ratio

Average Weekly Wage divided by Number of Dependents.

In [ ]:
for df in [X_train, X_val, df_test]:
    df['Wage/Dependents Ratio'] = df.apply(
        lambda row: round(row['Average Weekly Wage'] / row['Number of Dependents'], 2)
        if row['Number of Dependents'] > 0 else (row['Average Weekly Wage'])*2,
        axis=1
    )

In [ ]:
X_train[['Average Weekly Wage', 'Number of Dependents', 'Wage/Dependents Ratio']].head()

### Multiple Form Submissions

Flag if IME-4 Count > 1.

In [ ]:
for df in [X_train, X_val, df_test]:
    df['Multiple Form Submissions'] = (df['IME-4 Count'] > 1).astype(int)

In [ ]:
X_train[['IME-4 Count', 'Multiple Form Submissions']].head()

### Above County Wage Average

Indicates whether the row's wage is above the average for its county of injury.

In [ ]:
county_avg_wage_train = X_train.groupby('County of Injury')['Average Weekly Wage'].transform('mean')

X_train['Above County Wage Average'] = (X_train['Average Weekly Wage'] > county_avg_wage_train).astype(int)

county_avg_wage_val = X_val['County of Injury'].map(X_train.groupby('County of Injury')['Average Weekly Wage'].mean())
county_avg_wage_test = df_test['County of Injury'].map(X_train.groupby('County of Injury')['Average Weekly Wage'].mean())

X_val['Above County Wage Average'] = (X_val['Average Weekly Wage'] > county_avg_wage_val).astype(int)
df_test['Above County Wage Average'] = (df_test['Average Weekly Wage'] > county_avg_wage_test).astype(int)

In [ ]:
X_train[['County of Injury', 'Average Weekly Wage', 'Above County Wage Average']].head(10)

## Fix data types

In [ ]:
X_train.info()

First, we will transform 'Alternative Dispute Resolution', 'Attorney/Representative' and 'COVID-19 Indicator' into binary variables:

In [ ]:
def transform_columns_to_binary(dfs, columns):
    for df in dfs:
        for column in columns:
            if column in df.columns:
                df[column] = df[column].map({'N': 0, 'Y': 1})
    return dfs

In [ ]:
binary_columns = ['Alternative Dispute Resolution', 'Attorney/Representative', 'COVID-19 Indicator']
dfs = [X_train, X_val, df_test]


X_train, X_val, df_test = transform_columns_to_binary(dfs, binary_columns)

Now, we will fix data types:

In [ ]:
int_cols = [
    'Alternative Dispute Resolution', 'Attorney/Representative',
    'Age at Injury', 'Birth Year', 'IME-4 Count',
    'Agreement Reached', 'Number of Dependents', 
    'C-3 Received', 'First Hearing Happened', 'COVID-19 Indicator', 
    'Time to Assembly', 'Time to C-2', 
    'Multiple Form Submissions', 'Above County Wage Average'
]

float_cols = ['Average Weekly Wage', 'Wage/Dependents Ratio']

str_cols = [
    'Carrier Name', 'Carrier Type', 'County of Injury', 'Industry Code', 
    'WCIO Cause of Injury Code', 'WCIO Nature of Injury Code', 
    'WCIO Part Of Body Code', 'District Name', 
    'Gender', 'Medical Fee Region', 'Zip Code', 'Injury Day of Week'
] 

In [ ]:
def cast_columns(df):

    for col in int_cols:
        if col in df.columns:  
            df[col] = df[col].astype('Int64')  
  
    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype('float')
    
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype('string')
    
    return df

In [ ]:
X_train = cast_columns(X_train)
X_val = cast_columns(X_val)
df_test = cast_columns(df_test)

In [ ]:
X_train.info()

## Tranform datetime collumns into integers

Most machine learning models, especially classical ones, such as logistic regression and decision trees, do not handle datetime variables directly. For that reason, their transformation to numerical format was done by converting them into a ymd (year-month-day) format. 

In [ ]:
def process_dates(df):

    df['Accident_year'] = df['Accident Date'].dt.year
    df['Accident_month'] = df['Accident Date'].dt.month
    df['Accident_day'] = df['Accident Date'].dt.day
    
    df['Assembly_year'] = df['Assembly Date'].dt.year
    df['Assembly_month'] = df['Assembly Date'].dt.month
    df['Assembly_day'] = df['Assembly Date'].dt.day
    
    df['C2_year'] = df['C-2 Date'].dt.year
    df['C2_month'] = df['C-2 Date'].dt.month
    df['C2_day'] = df['C-2 Date'].dt.day
    
    df['Accident_ymd'] = df['Accident_year'] * 10000 + df['Accident_month'] * 100 + df['Accident_day']
    df['Assembly_ymd'] = df['Assembly_year'] * 10000 + df['Assembly_month'] * 100 + df['Assembly_day']
    df['C2_ymd'] = df['C2_year'] * 10000 + df['C2_month'] * 100 + df['C2_day']
    
    df = df.drop(columns=['Accident Date', 'Assembly Date', 'C-2 Date',
                          'Accident_year', 'Accident_month', 'Accident_day', 
                          'Assembly_year', 'Assembly_month', 'Assembly_day',
                          'C2_year', 'C2_month', 'C2_day'])
    return df

In [ ]:
X_train = process_dates(X_train)
X_val = process_dates(X_val)
df_test = process_dates(df_test)

In [ ]:
X_train.head()

In [ ]:
X_train.info()

In [ ]:
#Save X_train to a CSV file
X_train.to_csv('X_train_preprocessed.csv', index=False)

#Save X_val to a CSV file
X_val.to_csv('X_val_preprocessed.csv', index=False)

#Save df_test to a CSV file
df_test.to_csv('df_test_preprocessed.csv', index=False)

#Save y_train to a CSV file
y_train.to_csv('y_train.csv', index=False)

#Save y_val to a CSV file
y_val.to_csv('y_val.csv', index=False)

#Save df_test indexes to a CSV file
keep_identifier.to_csv('df_test_indexes.csv')

## Encoding

In [ ]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train) 
y_val = label_encoder.fit_transform(y_val) 

In [ ]:
cols_to_enc = [
    'Carrier Name', 'Carrier Type', 'County of Injury', 'Industry Code', 
    'WCIO Cause of Injury Code', 'WCIO Nature of Injury Code', 
    'WCIO Part Of Body Code', 'District Name', 
    'Gender', 'Medical Fee Region', 'Zip Code', 'Injury Day of Week'
] 

In [ ]:
target_encoder = ce.TargetEncoder(cols=cols_to_enc)

X_train[cols_to_enc] = target_encoder.fit_transform(X_train[cols_to_enc], y_train)
X_val[cols_to_enc] = target_encoder.transform(X_val[cols_to_enc])
df_test[cols_to_enc] = target_encoder.transform(df_test[cols_to_enc])

Since most of the machine learning models require the target variable to be numeric, we applied label encoding to the target variable. Label encoding converts categorical labels into numerical values so that the model can understand and process them. Each unique category in the target variable was mapped with an integer value for model training.

Target encoding was done for variables of object type. In this technique, each category of a categorical feature is replaced with the mean of the target variable within that particular category. Target encoding allows the model to learn the relationships between the categorical feature and the target well, which can improve the performance of models, especially with high cardinality categorical variables.

In [ ]:
X_train.info()

In [ ]:
X_val.info()

In [ ]:
df_test.info()

## Scalling

Now that all the variables are in numeric format, we go ahead and scale the data. Scaling is a must to ensure features contribute equally to the model performance when features are in different units or magnitudes. Because the scaler is fitted on the train data alone, the model never gets a prior knowledge about the test set- therefore keeping the wholesomeness of the evaluation process. Once fitted, the same scaling parameters will be applied to the training and testing datasets.

In [ ]:
#Add the 'Agreement Reached' column with 0s to df_test
df_test['Agreement Reached'] = 0

#Reorder columns to place 'Agreement Reached' in index 18
cols = df_test.columns.tolist()
cols.insert(18, cols.pop(cols.index('Agreement Reached')))  
df_test = df_test[cols]

In [ ]:
columns=X_train.columns

In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)

In [ ]:
X_val = scaler.transform(X_val)
df_test = scaler.transform(df_test)

In [ ]:
X_train = pd.DataFrame(X_train, columns=columns)
X_val= pd.DataFrame(X_val, columns=columns)
df_test = pd.DataFrame(df_test, columns=columns)

In [ ]:
X_train.head()

In [ ]:
X_val.head()

In [ ]:
df_test.head()

In [ ]:
y_train = pd.DataFrame(y_train)
y_val = pd.DataFrame(y_val)

In [ ]:
#Save X_train to a CSV file
X_train.to_csv('X_train_enc_scal.csv', index=False)

#Save X_val to a CSV file
X_val.to_csv('X_val_enc_scal.csv', index=False)

#Save df_test to a CSV file
df_test.to_csv('df_test_enc_scal.csv', index=False)

#Save y_train to a CSV file
y_train.to_csv('y_train_enc.csv', index=False)

#Save y_val to a CSV file
y_val.to_csv('y_val_enc.csv', index=False)

In [ ]:
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as 'scaler.pkl'")

joblib.dump(target_encoder, 'target_encoder.pkl')
print("Target Encoder saved as 'target_encoder.pkl'")

joblib.dump(label_encoder, 'label_encoder.pkl')
print("Label Encoder saved as 'label_encoder.pkl'")